# 08 — PCA dhe reduktimi i dimensionalitetit

        **Pyetja udhëheqëse:** Si gjejmë drejtimet ku të dhënat fizike ndryshojnë më shumë?

        Ky notebook është ndërtuar si një mësim i plotë kompjuterik. Rruga kryesore ndjek:
        **problem konkret → model matematikor → zbatim nga parimet e para → bibliotekë profesionale → krahasim → interpretim**.

        ## Objektivat
        - Të ndërtohet matrica e kovariancës.
- Të nxirren vlerat/vetorët vetjakë.
- Të kuptohet varianca e shpjeguar dhe rikonstruksioni.
- Të implementohet PCA manualisht.

        > **Rruga bazë për orën mësimore (45–60 min)** shënohet me seksionet **CORE**.  
        > Seksionet **EXTENDED** dhe **PLAYGROUND** janë për thellim dhe punë të pavarur.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import time

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("08 — PCA dhe reduktimi i dimensionalitetit")
print("Seed:", RANDOM_STATE)

# 1. Spektra të korreluara — CORE

In [ ]:
wl=np.linspace(400,750,120)
n=260
a=rng.normal(1.0,0.12,n)
b=rng.normal(0.5,0.09,n)
spectra=[]
for ai,bi in zip(a,b):
    s=ai*np.exp(-0.5*((wl-520)/28)**2)+bi*np.exp(-0.5*((wl-640)/35)**2)
    s+=rng.normal(0,0.02,len(wl))
    spectra.append(s)
X=np.array(spectra)
fig,ax=plt.subplots()
for i in range(20):
    ax.plot(wl,X[i],alpha=0.5)
ax.set_xlabel("Gjatësia valore [nm]"); ax.set_ylabel("Intensitet")
ax.set_title("Shembuj spektrash")
plt.show()

# 2. Nga kovarianca te drejtimet kryesore — CORE

Pas qendrimit $X_c=X-\bar X$, matrica e kovariancës është
$$
C=\frac{1}{N-1}X_c^TX_c.
$$
Drejtimet kryesore plotësojnë
$$
C\mathbf v_i=\lambda_i\mathbf v_i.
$$
Vlera vetjake $\lambda_i$ mat variancën përgjatë komponentit përkatës.

In [ ]:
Xc=X-X.mean(axis=0)
C=np.cov(Xc,rowvar=False)
vals,vecs=np.linalg.eigh(C)
idx=np.argsort(vals)[::-1]
vals,vecs=vals[idx],vecs[:,idx]
explained=vals/vals.sum()

fig,ax=plt.subplots()
ax.semilogy(np.arange(1,21),vals[:20],marker="o")
ax.set_xlabel("Komponenti"); ax.set_ylabel("Vlera vetjake")
ax.set_title("Spektri i vlerave vetjake")
plt.show()

### PCA nga paketa didaktike `src/`

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd()))
from src.pca.pca_from_scratch import PCAFromScratch
pca_src = PCAFromScratch(n_components=2).fit(X)
print("Varianca e shpjeguar (src):", np.round(pca_src.explained_variance_ratio_, 5))

# 3. Varianca kumulative — CORE

In [ ]:
fig,ax=plt.subplots()
ax.plot(np.cumsum(explained[:20]),marker="o")
ax.set_xlabel("Numri i komponentëve")
ax.set_ylabel("Varianca kumulative e shpjeguar")
ax.set_ylim(0,1.02)
ax.set_title("Sa dimensione duhen?")
plt.show()

# 4. Projekcioni në hapësirën PCA — CORE

In [ ]:
Z=Xc@vecs[:,:2]
fig,ax=plt.subplots()
scat=ax.scatter(Z[:,0],Z[:,1],c=a,s=16)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("Projekcioni PCA; ngjyra kodon amplitudën e pikut të parë")
plt.colorbar(scat,ax=ax,label="a")
plt.show()

# 5. Loadings dhe kuptimi fizik — CORE

In [ ]:
fig,ax=plt.subplots()
ax.plot(wl,vecs[:,0],label="PC1 loading")
ax.plot(wl,vecs[:,1],label="PC2 loading")
ax.set_xlabel("Gjatësia valore [nm]")
ax.set_ylabel("Koeficienti")
ax.legend(); ax.set_title("Çfarë strukturash spektrale kap PCA?")
plt.show()

# 6. Rikonstruksioni — CORE

In [ ]:
sample=7
fig,ax=plt.subplots()
ax.plot(wl,X[sample],label="origjinali")
for k in [1,2,5,15]:
    recon=X.mean(axis=0)+(Xc[sample]@vecs[:,:k])@vecs[:,:k].T
    ax.plot(wl,recon,label=f"k={k}")
ax.legend(); ax.set_xlabel("nm"); ax.set_ylabel("Intensitet")
ax.set_title("Rikonstruksioni me numër të ndryshëm komponentësh")
plt.show()

---
## Deri këtu: materiali bazë për orën mësimore

Në një seancë rreth 60 minuta, zakonisht ndalemi këtu. Seksionet që vijojnë janë për thellim,
ushtrime të pavarura dhe eksperimentim.

# 7. Krahasim me sklearn — EXTENDED

In [ ]:
from sklearn.decomposition import PCA
p=PCA(n_components=5).fit(X)
print("Manual:",np.round(explained[:5],5))
print("sklearn:",np.round(p.explained_variance_ratio_,5))

# 8. Aplikim jo-fizik: të dhëna biometrike — EXTENDED

Në biologji ose mjekësi, qindra matje të korreluara mund të reduktohen në pak drejtime dominante. Kjo lehtëson vizualizimin, por komponentët nuk janë automatikisht variabla kauzale.

# Ushtrime — PLAYGROUND

        ## A. Ushtrime të shkurtra
        1. Verifikoni ortogonalitetin e komponentëve të parë.
2. Llogaritni gabimin e rikonstruksionit ndaj k.
3. Standardizoni çdo gjatësi valore dhe përsëritni PCA.
4. Shpjegoni pse shenja e një eigenvector-i është arbitrare.

        ## B. Ushtrim i plotë
        Shtoni një pik spektral të tretë me amplitudë të pavarur dhe kontrolloni sa komponentë duhen për 95% variancë.

        ## C. Playground
        Ndryshoni numrin e mostrave, zhurmën, amplitudat dhe numrin e komponentëve.

In [ ]:
# TODO / PLAYGROUND
# Ndryshoni parametrat, shtoni grafikë ose krahasoni modele të tjera.